# Debugging clinic · Student

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 25. Debugging clinic

Find the reason a pandas result is wrong, then repair it.

Before you start: Modules 1–20; revisit selection, cleaning, and joins.

Official reference: https://pandas.pydata.org/docs/user_guide/copy_on_write.html

### M25C01. A label is not a row position

A table can use student IDs as its index. Position 0 still means the first row, but the label of that row is now S001. Read the index before choosing a selector.

**Read:** What labels does df use now?

Your explanation: _write here_.

**Predict:** Why do both selections contain two rows?

Your explanation: _write here_.

**Explain:** Would df.loc[0] mean the first row here?

Your explanation: _write here_.



In [ ]:
# Predict before running: Why do both selections contain two rows?
df = students().set_index("Student_ID")
by_position = df.iloc[0:2][["Name", "Score"]]
by_label = df.loc["S001":"S002", ["Name", "Score"]]
print(by_position)
print("Same rows:", by_position.equals(by_label))

**Your turn:** Select S003 through S005 by label, then select the same three students by position. Print both and confirm equality.

For the check, keep these result names: `by_label`, `by_position`.

**Hint:** The first selected row has position 2. The excluded stop must be one position after S005.

**Watch for this:** An integer inside loc is a label, even when it looks like a row number.

In [ ]:
# Change this code to complete the task.
# Question: Select S003 through S005 by label, then select the same three students by position. Print both and confirm equality.
df = students().set_index("Student_ID")
by_position = df.iloc[0:2][["Name", "Score"]]
by_label = df.loc["S001":"S002", ["Name", "Score"]]
print(by_position)
print("Same rows:", by_position.equals(by_label))

**Quick check:** Which stop is excluded?

1. The stop in an iloc slice
2. The stop in every loc label slice
3. Both stops are always excluded

**Explain what you learned:** Choose a selector from the meaning of the request, not from habit.

### M25C02. Repair a condition with two rules

A filter must produce one True or False value per row. Combine pandas conditions with & for AND or | for OR, with parentheses around each comparison. The broken expression is a comment so this example can run.

**Read:** What does each True in mask represent?

Your explanation: _write here_.

**Predict:** Will Lina be included?

Your explanation: _write here_.

**Explain:** Why is and wrong for these Series?

Your explanation: _write here_.



In [ ]:
# Predict before running: Will Lina be included?
df = students()
# Broken: df[(df["Score"] >= 80) and (df["Attendance"] >= 90)]
mask = (df["Score"] >= 80) & (df["Attendance"] >= 90)
result = df.loc[mask, ["Student_ID", "Score", "Attendance"]]
print(result)

**Your turn:** Find students whose Score is below 70 OR whose Attendance is below 80. Print their IDs and both values.

For the check, keep these result names: `result`.

**Hint:** Use | and keep both comparisons in parentheses.

**Watch for this:** Do not remove the parentheses around each comparison.

In [ ]:
# Change this code to complete the task.
# Question: Find students whose Score is below 70 OR whose Attendance is below 80. Print their IDs and both values.
df = students()
# Broken: df[(df["Score"] >= 80) and (df["Attendance"] >= 90)]
mask = (df["Score"] >= 80) & (df["Attendance"] >= 90)
result = df.loc[mask, ["Student_ID", "Score", "Attendance"]]
print(result)

**Quick check:** A student meets only one of two conditions. Which operator keeps that student?

1. &
2. |
3. and

**Explain what you learned:** Translate AND, OR, and NOT into &, |, and ~.

### M25C03. Change the table you actually intend to change

A filtered table and the original table are different objects. Use a single .loc assignment when the original must change. Use .copy() when creating a separate working table, and inspect both afterward.

**Read:** Which object receives the assignment?

Your explanation: _write here_.

**Predict:** What score will S004 have now?

Your explanation: _write here_.

**Explain:** Why should you avoid the commented chained assignment?

Your explanation: _write here_.



In [ ]:
# Predict before running: What score will S004 have now?
df = students()
mask = df["Student_ID"].eq("S004")
# Avoid: df[mask]["Score"] = 70
df.loc[mask, "Score"] = 70
result = df.loc[mask, ["Student_ID", "Score"]]
print(result)

**Your turn:** Start with a fresh df = students() and create a separate copy of its RAK rows named rak. Add 5 points to scores in rak. Show that Amira's copied score is 93 while her score in df remains 88.

For the check, keep these result names: `df`, `rak`.

**Hint:** Start with df.loc[df['Campus'].eq('RAK')].copy().

**Watch for this:** A method returning a new object is not an instruction to update the old one.

In [ ]:
# Change this code to complete the task.
# Question: Start with a fresh df = students() and create a separate copy of its RAK rows named rak. Add 5 points to scores in rak. Show that Amira's copied score is 93 while her score in df remains 88.
df = students()
mask = df["Student_ID"].eq("S004")
# Avoid: df[mask]["Score"] = 70
df.loc[mask, "Score"] = 70
result = df.loc[mask, ["Student_ID", "Score"]]
print(result)

**Quick check:** Which expression clearly updates df?

1. df[mask]['Score'] = 70
2. df.loc[mask, 'Score'] = 70
3. df.loc[mask].copy()['Score'] = 70

**Explain what you learned:** Make every data change explicit.

### M25C04. A correct total can still hide a bad merge

Two tables may contain the same key more than once. Matching repeated keys can multiply records. Validate the intended relationship and inspect unmatched IDs before calculating a report.

**Read:** What does validate='one_to_one' check?

Your explanation: _write here_.

**Predict:** Which IDs will be unmatched?

Your explanation: _write here_.

**Explain:** Why is a left merge alone insufficient to find extra assessment IDs?

Your explanation: _write here_.



In [ ]:
# Predict before running: Which IDs will be unmatched?
left = students()[["Student_ID", "Name"]]
right = assessment()
result = left.merge(right, on="Student_ID", how="outer",
                    validate="one_to_one", indicator=True)
print(result.loc[result["_merge"].ne("both"), ["Student_ID", "_merge"]])
print("Rows:", len(result))

**Your turn:** Keep only enrolled students with a left merge stored in result. Keep the merge indicator column, and store the count of students with no assessment in missing_count.

For the check, keep these result names: `missing_count`, `result`.

**Hint:** Use how='left', validate='one_to_one', and indicator=True.

**Watch for this:** Do not remove duplicate keys without understanding why they exist.

In [ ]:
# Change this code to complete the task.
# Question: Keep only enrolled students with a left merge stored in result. Keep the merge indicator column, and store the count of students with no assessment in missing_count.
left = students()[["Student_ID", "Name"]]
right = assessment()
result = left.merge(right, on="Student_ID", how="outer",
                    validate="one_to_one", indicator=True)
print(result.loc[result["_merge"].ne("both"), ["Student_ID", "_merge"]])
print("Rows:", len(result))

**Quick check:** What can a many-to-many merge do?

1. Always delete duplicates
2. Guarantee one row per student
3. Create multiple matches for each repeated key

**Explain what you learned:** Audit the join before trusting the summary.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates